In [1]:
###These are the same commands , though executed to a csv file.

In [24]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import csv

df=pd.read_csv('../data/finaccess2024_datasprint.csv',quoting= csv.QUOTE_ALL,skipinitialspace=True)


##Getting the number of rows and columns
print("=" *20)
print("SHAPE")
print("="*20)
print(f"Rows:{df.shape[0]},Columns:{df.shape[1]}")

##Checking for data types
print("\n"+"="*20)
print("COLUMNS AND DATA TYPES")
print("="+"="*20)
print(df.dtypes.to_string())


#First and Last few rows
print("\n"+"=" *20)
print("FIRST 5 ROWS")
print("="*20)
print(df.head(5).to_string())

print("\n"+"=" *20)
print("LAST 5 ROWS")
print("="*20)
print(df.tail().to_string())


##Basic statistics
print("\n"+"=" *20)
print("Basic Statistics")
print("=" *20)
print(df.describe().to_string())

##Missing Values
print("\n"+"=" *20)
print("MISSING VALUES")
print("=" *20)
missing = df.isnull().sum()
missing_pct =(df.isnull().mean()*100).round(2)
missing_df=pd.DataFrame({"Count":missing, "Percent":missing_pct})
print(missing_df[missing_df["Count"] > 0].sort_values("Percent", ascending=False).to_string())
if missing_df[missing_df["Count"]>0].empty:
    print("No missing values!")

##Checking for duplicate rows
print("\n"+"=" *20)
print("DUPLICATES")
print("=" *20)
print(f"Duplicate rows:{df.duplicated().sum}") 

##Checking for unique values
print("\n"+"=" *20)
print("Unique values per categorical column")
print("=" *20)
for col in df.select_dtypes(include=['object','str','category']).columns:
    print(f"\n---{col} ({df[col].nunique()} unique)-----")
    print(df[col].value_counts().head(10).to_string())



print("\n"+"="*20)
print("Numerical Values- Distributions")
print("="*20)
for col in df.select_dtypes(include=[np.number]).columns:
    print(f"\n---{col}---")
    print(f"Min: {df[col].min} \n Max: {df[col].max()}, \n Mean: {df[col].mean():.2f}, \n Median: {df[col].median():.2f} , \n Std: {df[col].std():.2f}")

##Checking for target variable
print("\n"+"="*20)
print("TARGET VARIABLE:financial_status")
print("="*20)
if "financial status" in df.columns:
    print(df["financial_status"].value_counts())
    print(df["financial_status"].value_counts(normalize=True).round(3)*100)
else:
    print("Column 'financial_status' not found")
    print("Available columns:",list(df.columns))

education_level unique values after reload:
  '"Don\'t know (DO NOT READ OUT)"'                            count=2
  '"None "'                                                    count=3078
  '"Other (Specify) "'                                         count=2
  '"Primary completed"'                                        count=3953
  '"Refused to Answer (DO NOT READ OUT)"'                      count=3
  '"Secondary completed "'                                     count=4113
  '"Some primary "'                                            count=3731
  '"Some secondary"'                                           count=2731
  '"University completed "'                                    count=988
  '95'                                                         count=2
  'Completed technical training after secondary school'        count=1421
  'Some technical training after secondary school'             count=501
  'Some university'                                            count=346
SHAPE
Row

In [30]:
#Cleaning the data
##Shape of raw data before cleaning

print(f"Loaded raw data:{df.shape[0]} rows, {df.shape[1]} columns")


##Finding and Dropping Duplicates
dupes_before = df.duplicated().sum()
df=df.drop_duplicates()
print(f"\n STEP 1 - Duplicates Found:{dupes_before}, dropped them.\n Rows now:{df.shape[0]}")

##Cleaning Junk Responses
#Education Level
junk_education =[
    'Refused to Answer (DO NOT READ OUT)',
    "Don't know(DO NOT READ OUT)",
    'Other (Specify)',
    '95'
]
rows_before=len(df)
df = df[~df['education_level'].isin(junk_education)]
rows_dropped= rows_before - len(df)
print(f"\n Step 2- educational level: Dropped {rows_dropped} junk rows.")
print(f"\nClean  values:{sorted(df['education_level'].unique())}")

##marital status 
junk_marital=[
    "Don't Know   (DO NOT READ OUT)",
    "Refused to Answer(DO NOT READ OUT)"
]
rows_before= len(df)
df=df[~df['marital_status'].isin((junk_marital))]
rows_dropped= rows_before - len(df)
print(f"\n Step 3 - marital_status :Dropped {rows_dropped} junk rows.")
print(f"\nClean values:{sorted(df['marital_status'].unique())}")


##Filling in blank values in Barriers_bank
missing_before = df['barriers_bank'].isnull().sum()
df['barriers_bank']= df['barriers_bank'].fillna('No barrier')
print(f"\nStep 4 - barriers_bank: Filled {missing_before} NaN values with 'No barrier'.")

##Renaming '0' in barriers_mobile_money for clarity
count_zero= (df['barriers_mobile_money'] == '0').sum()
df['barriers_mobile_money'] = df['barriers_mobile_money'].replace('0','No barrier')
print(f"\nStep 5 - barriers_mobile_money: Rename {count_zero} '0' values to 'No barrier' ")

##Removing embedded quotes from ALL text columns
def clean_quotes(val):
    if isinstance(val, str):
        val = val.strip()          
        val = val.strip('"')        
        val = val.strip("'")        
        val = val.strip()           
    return val


for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].apply(clean_quotes)


##Verification
print("\n"+"=" *20)
print("VERIFICATION")
print("\n"+"=" *20)

missing = df.isnull().sum()
missing_total= missing.sum()
print(f"Missing values: {missing_total}")
if missing_total >0:
    print("WARNING- still have missing values:")
    print(missing[missing>0])

dupes=df.duplicated().sum()
print(f"Duplicated Rows: {dupes}")

print(f"Final Shape: {df.shape[0]} rows, {df.shape[1]} columns")

print(f"Target Variable (financial_status)")
print(df['financial_status'].value_counts())
print()
print((df['financial_status'].value_counts(normalize=True) * 100).round(2))


print(f"\nAll columns — unique value counts:")
for col in df.columns:
    print(f"  {col}: {df[col].nunique()} unique")


print(f"Education level clean values")
for v in sorted(df['education_level'].unique()):
    print(f"{repr(v):60s} count={int((df['education_level']==v).sum())}")


output_path = '../data/finaccess2024_cleaned.csv'
df.to_csv(output_path, index=False)
print(f"\n{'=' * 50}")
print(f"SAVED cleaned data to: {output_path}")
print(f"  Rows: {df.shape[0]}, Columns: {df.shape[1]}")
print(f"{'=' * 50}")

Loaded raw data:20859 rows, 28 columns

 STEP 1 - Duplicates Found:0, dropped them.
 Rows now:20859

 Step 2- educational level: Dropped 0 junk rows.

Clean  values:['Completed technical training after secondary school', "Don't know (DO NOT READ OUT)", 'None', 'Primary completed', 'Secondary completed', 'Some primary', 'Some secondary', 'Some technical training after secondary school', 'Some university', 'University completed']

 Step 3 - marital_status :Dropped 4 junk rows.

Clean values:['Divorced/separated', "Don't know   (DO NOT READ OUT)", 'Married/Living with partner', 'Single/Never Married', 'Widowed']

Step 4 - barriers_bank: Filled 0 NaN values with 'No barrier'.

Step 5 - barriers_mobile_money: Rename 0 '0' values to 'No barrier' 


C:\Users\jerem\AppData\Local\Temp\ipykernel_27512\3136850775.py:58: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df.select_dtypes(include='object').columns:



VERIFICATION

Missing values: 0
Duplicated Rows: 0
Final Shape: 20855 rows, 28 columns
Target Variable (financial_status)
financial_status
Worsened           10967
Stayed the same     5607
Improved            4281
Name: count, dtype: int64

financial_status
Worsened           52.59
Stayed the same    26.89
Improved           20.53
Name: proportion, dtype: float64

All columns — unique value counts:
  county: 47 unique
  location_type: 2 unique
  Sex: 2 unique
  Age: 6 unique
  household_size: 20 unique
  education_level: 10 unique
  marital_status: 5 unique
  monthly_income: 236 unique
  Savings_formal: 2 unique
  Savings_informal: 2 unique
  Loan_formal: 2 unique
  Loan_informal: 2 unique
  defaulted: 2 unique
  formal_service_use: 2 unique
  mobile_money_access: 2 unique
  barriers_mobile_money: 10 unique
  mobile_ownership_1: 2 unique
  experienced_shock: 2 unique
  nfhi_11: 2 unique
  nfhi_12: 2 unique
  nfhi_13: 2 unique
  accessto_13k_1month: 2 unique
  not_difficult: 2 unique
 

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
##Loading the clean data
df=pd.read_csv('../data/finaccess2024_cleaned.csv')
print(f" Clean data loaded : {df.shape[0]} rows {df.shape[1]} columns")
